# Pause-based Metrics:

## Module 05: Lesson 01

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Critt-Kent/CRITT-Academy/blob/main/modules/05_Advanced_Analysis/01_Pause_based_Metrics.ipynb) | [This video linked here](https://vimeo.com/1201603642) explains how to use this code notebook in Google Colab 🤠

### What You Will Do In This Lesson

1. Install and import all necessary Python libraries
2. Load Segment and Keystroke Data tables (SG and KD tables) from the TPR-DB for a single study (a public study) and then read them into a DataFrame
3. Add pause-based metrics based on a different pause threshold than is available by default in the SG tables
4. Calculate additional, derived pause-based metrics such as pause-to-word ratio and average pause ratio

### First time working with a CRITT Academy code notebook?

If you haven't yet gone through the [CRITT Academy Module 01, Lesson 01, “Code Notebooks and the TPR_DB”](https://github.com/Critt-Kent/CRITT-Academy/blob/main/modules/01_Foundations/01_Notebooks_and_TPRDB.ipynb), you should do that first (this will help you understand Steps 1 through 3 much better).

## Step 1. Import all necessary Python libraries

- **NumPy**: `numpy` is used for scientific computing and data science
- **Pandas**: `pandas` is used for data manipulation, organization, and analysis
- **SciPy `stats`**: the `stats` module within `scipy` is used for statistical analysis and probability distributions
- **Matplotlib**: `matplotlib` is used for data visualization
- **Seaborn**: `seaborn` is used for nice-looking statistical graphics
- **tprdb-utilities**: `tprdb_utilities` is used for getting data remotely from the CRITT TPR-DB and then for reading data tables from a certain study (or studies) into a Pandas DataFrame.

In [ ]:
# Environment Setup
import sys
from importlib.metadata import version, PackageNotFoundError
from packaging.version import Version

# Enforce a minimum Python version of 3.9+
if sys.version_info < (3, 9):
    raise RuntimeError(
        f"❌ Python 3.9 or higher is required. "
        f"You are currently running Python {sys.version_info.major}.{sys.version_info.minor}."
    )

REQUIRED = {
    "numpy": "2.4.0",
    "pandas": "3.0.0",
    "scipy": "1.17.0",
    "matplotlib": "3.10.0",
    "seaborn": "0.13.0",
    "tprdb-utilities": "0.5.0",
}

def _check_versions():
    outdated = []
    for pkg, min_ver in REQUIRED.items():
        try:
            installed = version(pkg)
            if Version(installed) < Version(min_ver):
                outdated.append(f"  {pkg}: installed {installed}, need >={min_ver}")
        except PackageNotFoundError:
            raise ImportError(f"{pkg} is not installed")
    return outdated

# Check if dependencies are missing and install them automatically
try:
    import numpy as np
    import pandas as pd
    import scipy.stats
    import matplotlib.pyplot as plt
    import seaborn as sns
    import tprdb_utilities

    outdated = _check_versions()
    if outdated:
        raise ImportError("Outdated packages:\n" + "\n".join(outdated))
    else:
        print("🤠 All core dependencies are already installed. You are ready to go! 🤘")
except ImportError:
    print("Missing dependencies detected. Installing required packages...")

    try:
        # The %pip magic ensures installation happens in the active Jupyter kernel
        %pip install "numpy>=2.4.0" "pandas>=3.0.0" "scipy>=1.17.0" "matplotlib>=3.10.0" "seaborn>=0.13.0" "tprdb-utilities>=0.5.0"

        print("\n🤠 Installation complete 🤘 If imports fail on the next cell, please restart the kernel.")
    except Exception as e:
        print(f"❌ An error occurred during installation: {e}")
        print("If using Google Colab, you may just have to restart your runtime now to use the newly installed packages.")

  Attempting uninstall: tprdb_utilities
    Found existing installation: tprdb-utilities 0.4.0
    Uninstalling tprdb-utilities-0.4.0:
      Successfully uninstalled tprdb-utilities-0.4.0
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Now, import the libraries

try:
    # Standard Python library imports
    import numpy as np
    import pandas as pd
    import scipy.stats
    import matplotlib
    import matplotlib.pyplot as plt
    import seaborn as sns

    # TPR-DB utilities import
    from tprdb_utilities import fetch_TPRDB_tables, read_TPRDB_tables, recompute_pause_based_metrics

    print("✅ All imports successful!")

except ImportError as e:
    print(f"❌ An error occurred during imports: {e}")
    print("Please ensure all dependencies are installed and the kernel is restarted if necessary.")

## Step 2: Get Segments and Keystroke Data Tables and load them into DataFrames

### 🫵 Change code cell below

To get ready to run the `fetch_TPRDB_tables()` function, change the code cell below depending on your situation:
- **If *NOT* using Google Colab**:
    - Follow the **1st** set of directions to change the value of the `mothership_clone_location` variable
- **If you *are* using Google Colab**:
    - Follow the **2nd** set of directions to
        1. Comment out the code from the 1st set of directions
        2. Uncomment the code from the 2nd set of directions
        3. mount your Google Drive (unless you want it in a more specific location, you shouldn't need to change the value of the `mothership_clone_location` variable after uncommenting the code)

[Link to 📽️ Video on How to Comment Out and Uncomment Code](https://vimeo.com/1201588460)

#### ⚠️ If you are unsure what is going on here..

If you are confused about this step, it means you should go back and review [CRITT Academy Module 01, Lesson 01, “Code Notebooks and the TPR-DB”](https://github.com/Critt-Kent/CRITT-Academy/blob/main/modules/01_Foundations/01_Notebooks_and_TPRDB.ipynb).


In [ ]:
# Commenting/Uncommenting Code: ⌘ + / (Mac) or Ctrl + / (Windows/Linux)

# 🫵 If NOT using Google Colab:
# 1) Change what's in the quotes below ("./") to point to where you want your mothership clone to be saved.
# 2) Then, run this code cell
mothership_clone_location = "./"

# 🫵 If using Google Colab:
# 1) Comment out the above line of code
# 2) Uncomment the lines of code below
# 3) Then, run this code cell

# from google.colab import drive
# drive.mount('/content/drive')
# mothership_clone_location = "/content/drive/MyDrive/"

In [ ]:
# fetch data tables (notice we are using the `mothership_clone_location` variable we defined in the cell above)

fetch_TPRDB_tables(
    path=mothership_clone_location,
    studies=["BML12"],
    extensions=["ss", "sg", "kd"],
    public=True,
)

In [ ]:
# read the Segments (SG) and Keystroke Data tables into a DataFrame

# this study uses (English) multiLing texts translated into Spanish
studies=["BML12"]

SG = read_TPRDB_tables(
    path=f'{mothership_clone_location}/tprdb-mothership-clone',
    user='PUBLIC',
    studies=studies,
    extension="sg",
    verbose=1,
)

KD = read_TPRDB_tables(
    path=f'{mothership_clone_location}/tprdb-mothership-clone',
    user='PUBLIC',
    studies=studies,
    extension="kd",
    verbose=1,
)

Reading: BML12	with 184 'sg' Tables
Total 'sg' data rows: 1229, columns: 39
Reading: BML12	with 184 'kd' Tables
Total 'kd' data rows: 93138, columns: 25


### Take a look at the pause-based metrics

In this next code cell, you'll take a look at *everything* in the Segments (SG) and Keystroke Data (KD) DataFrames, but take a *close look* at the Columns in the Segments DataFrame that start with these letters:

* **TB** – **T**yping **B**ursts (the number of stretches of typing used to translate/edit each segment)
* **TG** – **T**yping **G**ap (amount of time, in milliseconds, spent pausing between typing bursts)
* **TD** – **T**yping **D**uration (amount of time, in milliseconds, spent typing)

All of the preceding metrics depend on a certain **pause threshold**. We have to define how long someone has to **not be typing** in order for it to be considered a pause. There are three pause thresholds that come built in to the Segments (SG) tables:

* **1000**: 1000 milliseconds (nice and arbitrary 🤓)
* **KUI**: Keystroke Unit Interruption
* **PUB**: Production Unit Break

To read more about **KUI** and **PUB**, see the [TPR-DB Documentation: Features List](https://critt-kent.github.io/TPR-DB-documentation/analyze/features/).

In [ ]:
# Check the two DataFrames' shapes and display the first few rows (with all columns visible) of each DataFrame
print(f"📊 The Segments DataFrame has {SG.shape[0]} rows and {SG.shape[1]} columns")

with pd.option_context("display.max_columns", None):
    display(SG.head())

print(f"📊 The Keystroke Data table has {KD.shape[0]} rows and {KD.shape[1]} columns")

with pd.option_context("display.max_columns", None):
    display(KD.head())

## Step 3: Recompute the pause-based metrics based on a custom pause threshold

We can use a function called `recompute_pause_based_metrics()` (from the `transformer` module of the [`tprdb-utilities` python library](https://pypi.org/project/tprdb-utilities/)) to add new Typing Burst, Typing Gap, and Typing Duration metrics that are based on a custom pause threshold that we define to the Segments (SG) DataFrame.

In [ ]:
# view help documentation first
recompute_pause_based_metrics?

In [ ]:
# add pause-based metrics using a pause threshold of 500 milliseconds
SG = recompute_pause_based_metrics(SG, KD, 500)

In [ ]:
# view the columns that were added to the Segments DataFrame
with pd.option_context("display.max_columns", None):
    display(SG.head())

,Id,Study,Session,StudySession,SL,TL,Task,Text,Part,KUI,PUB,STseg,TTseg,LenS,LenT,TokS,TokT,Ins,Del,Dur,FixS,TrtS,FixT,TrtT,Nedit,PreGap,PostGap,TB1000,TG1000,TD1000,TB_KUI,TG_KUI,TD_KUI,TB_PUB,TG_PUB,TD_PUB,String,HTot,HTotN,TB750,TG750,TD750
0,1,BML12,P11_T1,BML12@P11_T1,en,es,T,1,P11,216,468,1,1,41,58,6,9,74,15,12265,0,0,12,2544,0,6766,2985,2,1235,11030,9,4313,7952,5,3048,9217,Condenan_a_un_enfermero_asesino_a_cuatro_caden...,1.5444,0.3117,3,2048,10217
1,2,BML12,P11_T1,BML12@P11_T1,en,es,T,1,P11,216,468,2,2,99,127,18,25,130,5,37422,0,0,13,2431,1,2985,1484,6,18359,19063,22,24596,12826,10,21171,16251,Hoy_ha_sido_condenado_a_cadena_perpetua_el_enf...,0.8745,0.1765,7,19344,18078
2,3,BML12,P11_T1,BML12@P11_T1,en,es,T,1,P11,216,468,3,3,113,134,21,22,141,9,36313,0,0,27,3792,1,1484,11750,6,12796,23517,29,21562,14751,14,17453,18860,"Norris_,_de_32_años_y_originario_de_Glasgow_,_...",1.1939,0.2410,7,13624,22689
3,4,BML12,P11_T1,BML12@P11_T1,en,es,T,1,P11,216,468,4,4,81,81,16,15,135,54,38234,0,0,8,886,1,11750,187,5,12359,25875,36,23060,15174,14,17061,21173,Ayer_se_le_consideró_culpable_de_cuatro_casos_...,1.2368,0.2496,5,12359,25875
4,5,BML12,P11_T1,BML12@P11_T1,en,es,T,1,P11,216,468,5,5,65,77,14,15,87,11,13235,2,408,12,1375,1,187,1937,2,1235,12000,10,4797,8438,6,3907,9328,"Le_conenaron_a_cuatro_cadenas_perpetuas_,_una_...",1.2930,0.2610,3,2188,11047


## Step 4: Calculate derivative pause-based metrics

- $\text{Pause-to-Word Ratio (PWR)} = \frac{\text{number of pauses in segment}}{\text{number of words in segment}}$
- $\text{Average Pause Ratio (APR)} = \frac{\text{?????}}{\text{?????}}$

In [ ]:
# calculate different Pause-to-Word Ratios (PWR) using different pause thresholds and add them to the Segments (SG) DataFrame
SG["PWR1000"] = (SG["TB1000"] + 1) / SG["TokS"]
SG["PWR_KUI"] = (SG["TB_KUI"] + 1) / SG["TokS"]
SG["PWR_PUB"] = (SG["TB_PUB"] + 1) / SG["TokS"]
SG["PWR500"] = (SG["TB500"] + 1) / SG["TokS"]

# View new DataFrame with Pause-to-Word Ratios
with pd.option_context("display.max_columns", None):
    display(SG.head())